In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# RBFE many pairs in a workflow

End-to-end workflow for one relative binding free energy pair:

1. Load the BRD protein and two ligands, register them on the data platform
2. Run system prep in RBFE mode
3. Inspect the prepared system
4. Run RBFE FEP on the prepared system

## Setup

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Ligand,
    Protein,
    RBFE,
    RBFEParams,
    SystemPrep,
    PreparedSystem,
)
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()
client

## 1. Load structures and register on the data platform

We use the BRD4 example protein and two congeneric ligands from the bundled
dataset. `sync()` uploads files (when needed) and registers records on the
data platform.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()
protein.id

In [ ]:
ligands = []
for ligand_file in ["brd-2.sdf", "brd-3.sdf", "brd-4.sdf"]:
    ligand = Ligand.from_sdf(BRD_DATA_DIR / ligand_file)
    ligand.sync()
    ligands.append(ligand)
    

ligands

## 4. Run RBFE workflow on protein and ligand inputs

Submit FEP only (`mode="rbfe"`) using the prepared binding/solvation XML paths.
We quote first, then confirm to start the job.

`test_run=1` shortens the simulation for exploration; use `test_run=0` for
production-quality results.

In [ ]:
rbfe = RBFE(
    protein=protein,
    pairs=[(ligands[0], ligands[1]),
           (ligands[1], ligands[2]),
          ],
    params=RBFEParams(test_run=1),
)
rbfe

In [ ]:
rbfe.start(quote=True)
rbfe.estimate

In [ ]:
rbfe.confirm()

In [ ]:
task = await rbfe.watch()

## Results

In [ ]:
rbfe.get_results()

## Get user logs

Retrieve user logs generated by tools during this workflow

In [ ]:
rbfe.get_user_logs()

# Get prepared system and show

Use this to retrieve prepared systems and inspect them

In [ ]:
system = rbfe.get_prepared_system()
system.show()